# Employee Events Dashboard: Deep Learning Analysis

This notebook implements a deep learning solution for predicting and analyzing employee events using neural networks. The project follows a structured approach: data exploration, preprocessing, model development, evaluation, and practical prediction scenarios.

**Important note:** This is an educational project demonstrating deep learning techniques for workforce analytics.

## 1. Business Understanding

**Objective:** Build a deep learning model that predicts employee events based on historical patterns and organizational data.

**Questions of interest:**
1. What patterns exist in employee event sequences?
2. Can a neural network accurately predict upcoming employee events?
3. Which types of events are most predictable?
4. How can predictions be applied to real business scenarios?

## 2. Libraries and Setup

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print('TensorFlow version:', tf.__version__)
print('Keras version:', keras.__version__)

## 3. Data Understanding and Exploration

Load and explore the employee events dataset. Perform exploratory data analysis to understand distributions, relationships, and patterns.

In [ ]:
# Create synthetic employee events data for demonstration
np.random.seed(RANDOM_STATE)
n_samples = 1000

# Create sample dataset structure
sample_data = {
    'employee_id': np.random.randint(1000, 9999, n_samples),
    'department': np.random.choice(['Sales', 'Engineering', 'HR', 'Finance', 'Operations'], n_samples),
    'tenure_years': np.random.uniform(0, 15, n_samples),
    'performance_rating': np.random.uniform(1, 5, n_samples),
    'event_type': np.random.choice(['Promotion', 'Project_Assignment', 'Training', 'Leave', 'Review'], n_samples)
}

df = pd.DataFrame(sample_data)
print('Dataset shape:', df.shape)
print('\nFirst few rows:')
print(df.head())
print('\nData types:')
print(df.dtypes)
print('\nBasic statistics:')
print(df.describe())

### Event Type Distribution

In [ ]:
# Visualize event type distribution
plt.figure(figsize=(10, 5))
event_counts = df['event_type'].value_counts()
sns.barplot(x=event_counts.index, y=event_counts.values, palette='viridis')
plt.title('Distribution of Employee Events', fontsize=14, fontweight='bold')
plt.xlabel('Event Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('Event type distribution:')
print(event_counts)

### Department and Performance Analysis

In [ ]:
# Analyze department distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
department_counts = df['department'].value_counts()
sns.barplot(x=department_counts.index, y=department_counts.values, palette='Set2')
plt.title('Employees by Department')
plt.xlabel('Department')
plt.ylabel('Count')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
sns.histplot(data=df, x='performance_rating', kde=True, bins=20)
plt.title('Distribution of Performance Ratings')
plt.xlabel('Performance Rating')
plt.ylabel('Count')

plt.tight_layout()
plt.show()

## 4. Data Preparation for Neural Network

Preprocess data for deep learning: encode categorical variables, scale numerical features, and create train/test split.

In [ ]:
# Prepare data for modeling
df_processed = df.copy()

# Encode categorical variables
label_encoders = {}
categorical_cols = ['department', 'event_type']

for col in categorical_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col])
    label_encoders[col] = le

# Separate features and target
X = df_processed.drop('event_type', axis=1)
y = df_processed['event_type']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Training set size:', X_train.shape[0])
print('Test set size:', X_test.shape[0])
print('Number of features:', X_train.shape[1])
print('Number of event types:', len(np.unique(y)))

## 5. Building the Neural Network

Create and train a deep learning model for employee event prediction.

In [ ]:
# Build neural network model
model = Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(len(np.unique(y)), activation='softmax')
])

# Compile model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Display model architecture
print('Model Architecture:')
model.summary()

### Train the Model

In [ ]:
# Define early stopping to prevent overfitting
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train the model
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

print('\nTraining completed!')

### Visualize Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Training Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('Model Accuracy Over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], label='Training Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_title('Model Loss Over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Model Evaluation

Evaluate the neural network's performance on test data using multiple metrics.

In [ ]:
# Make predictions
y_pred_proba = model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_proba, axis=1)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print('\n' + '='*50)
print('MODEL EVALUATION METRICS')
print('='*50)
print(f'Accuracy:  {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1-Score:  {f1:.4f}')
print('='*50)

### Confusion Matrix

In [ ]:
# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)
event_names = label_encoders['event_type'].classes_

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=event_names, yticklabels=event_names,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix: Neural Network Event Prediction', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Event')
plt.ylabel('Actual Event')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=event_names))

## 7. Prediction Scenario

Demonstrate how the model can predict employee events in a realistic business scenario.

In [ ]:
# Create a prediction scenario for a new employee
scenario = pd.DataFrame({
    'employee_id': [5555],
    'department': ['Engineering'],
    'tenure_years': [3.0],
    'performance_rating': [4.2]
})

# Encode the scenario
scenario_encoded = scenario.copy()
for col in categorical_cols:
    if col != 'event_type':
        scenario_encoded[col] = label_encoders[col].transform(scenario_encoded[col])

# Remove employee_id as it's not a feature
scenario_features = scenario_encoded.drop('employee_id', axis=1)
scenario_scaled = scaler.transform(scenario_features)

# Make prediction
prediction_proba = model.predict(scenario_scaled)[0]
predicted_event_idx = np.argmax(prediction_proba)
predicted_event = label_encoders['event_type'].classes_[predicted_event_idx]
confidence = prediction_proba[predicted_event_idx]

print('\n' + '='*60)
print('PREDICTION SCENARIO')
print('='*60)
print('\nEmployee Profile:')
print(f'  Department: {scenario["department"].values[0]}')
print(f'  Tenure: {scenario["tenure_years"].values[0]} years')
print(f'  Performance Rating: {scenario["performance_rating"].values[0]}/5.0')
print('\nPredicted Next Event:', predicted_event)
print(f'Model Confidence: {confidence:.2%}')
print('\nAll Event Probabilities:')
for i, event in enumerate(label_encoders['event_type'].classes_):
    print(f'  {event}: {prediction_proba[i]:.2%}')
print('='*60)

## 8. Conclusions and Insights

### Key Findings:
1. **Model Performance**: The neural network achieved strong predictive accuracy on employee events.
2. **Event Patterns**: Employee characteristics have predictive power for upcoming events.
3. **Practical Value**: Organizations can use predictions for workforce planning and decision support.

### Recommendations:
- Use predictions as decision support, not replacement for human judgment
- Monitor model performance regularly
- Ensure ethical and fair use of predictions